# 06e - XAI Cross-Model Comparison
Krizove porovnani vysvetleni ze vsech metod a modelu: SHAP, LIME, ICE/c-ICE.

**Otazky na ktere odpovida:**
- Ktere featury jsou konzistentne dulezite u VSECH modelu (SHAP + LIME)?
- Jak se lisi pohled stromovych modelu (RF, GB) a linearniho modelu (LR)?
- Jsou featury s nejvetsim SHAP vlivem take ty, ktere vykazuji nejvetsi heterogenitu v ICE?
- Jsou nejdulezitejsi featury z XAI konzistentni s Feature Importance z baseline notebooku (033)?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

ctx       = joblib.load('../results/xai/xai_context.pkl')
shap_all  = joblib.load('../results/xai/shap_values_all.pkl')

X_test_prep  = ctx['X_test_prep']
tuned_models = ctx['tuned_models']

feature_names = X_test_prep.columns.tolist()
print(f"Nacten kontext: {len(feature_names)} promennych, {len(tuned_models)} modely")

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


Nacten kontext: 19 promennych, 4 modely


## SHAP: průměrná absolutní důležitost — normalizovaná
Normalizujeme na rozsah 0–1 v ramci kazdeho modelu pro fér porovnani.

In [2]:
shap_ranks = {}

for name, shap_vals in shap_all.items():
    mean_abs = pd.Series(
        np.abs(shap_vals.values).mean(axis=0),
        index=feature_names
    )
    shap_ranks[name] = mean_abs / mean_abs.max()  # normalizace 0-1

df_shap_norm = pd.DataFrame(shap_ranks)
df_shap_norm['SHAP_consensus'] = df_shap_norm.mean(axis=1)
df_shap_norm = df_shap_norm.sort_values('SHAP_consensus', ascending=False)

print("Top-10 featur podle SHAP konsensu (prumer pres modely):")
print(df_shap_norm['SHAP_consensus'].head(10).round(3).to_string())

Top-10 featur podle SHAP konsensu (prumer pres modely):
CGPA                   1.000
Stress_Index           0.251
Gender                 0.143
Scholarship            0.143
Attendance_Rate        0.134
Semester               0.098
Part_Time_Job          0.066
Family_Income          0.065
Internet_Access        0.061
Travel_Time_Minutes    0.055


## Heatmapa: konzistence napříč modely

In [3]:
top_features = df_shap_norm.head(15).index

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df_shap_norm.loc[top_features, list(tuned_models.keys())],
    annot=True, fmt='.2f', cmap='viridis',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Normalizovana SHAP dulezitost (0-1)'}
)
ax.set_title('Konzistence dulezitosti featur pres modely (SHAP)', fontsize=13, pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Srovnání: SHAP vs. Feature Importance (Random Forest)
Porovnavame globalní pohled SHAP s klasickou Feature Importance z notebooku 033.

In [4]:
rf_model = tuned_models['Random Forest']
rf_fi    = pd.Series(rf_model.feature_importances_, index=feature_names)
rf_fi_norm = rf_fi / rf_fi.max()

shap_rf_norm = df_shap_norm['Random Forest']

compare_df = pd.DataFrame({
    'Feature Importance (RF)': rf_fi_norm,
    'SHAP (RF)': shap_rf_norm,
}).sort_values('SHAP (RF)', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
x = np.arange(len(compare_df))
w = 0.38
ax.barh(x + w/2, compare_df['Feature Importance (RF)'], height=w, label='Feature Importance', color='#1D9E75', alpha=0.8)
ax.barh(x - w/2, compare_df['SHAP (RF)'],               height=w, label='SHAP (mean |shap|)', color='#378ADD', alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(compare_df.index, fontsize=10)
ax.set_xlabel('Normalizovana dulezitost (0-1)')
ax.set_title('Feature Importance vs. SHAP — Random Forest', fontsize=13, pad=15)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nKorelace mezi Feature Importance a SHAP (RF):")
corr = compare_df['Feature Importance (RF)'].corr(compare_df['SHAP (RF)'])
print(f"  Pearsonova korelace: {corr:.3f}")


Korelace mezi Feature Importance a SHAP (RF):
  Pearsonova korelace: 0.982


## Lineární vs. stromové modely: pohled na featury
Stromové modely (RF, GB) mohou zachytit nelinearni vztahy — zjistime, zda se jejich pohled lisi od linearni regrese.

In [5]:
lr_shap  = pd.Series(np.abs(shap_all['Logistic Regression'].values).mean(axis=0), index=feature_names)
rf_shap  = pd.Series(np.abs(shap_all['Random Forest'].values).mean(axis=0),        index=feature_names)
gb_shap  = pd.Series(np.abs(shap_all['Gradient Boosting'].values).mean(axis=0),   index=feature_names)

# Featury kde se LR a tree modely nejvice rozchazeji
avg_tree = (rf_shap + gb_shap) / 2
diff     = (avg_tree / avg_tree.max() - lr_shap / lr_shap.max()).abs().sort_values(ascending=False)

print("Featury s nejvetsi neshodu mezi LR a stromovymi modely:")
print(diff.head(8).round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
diff.head(10).sort_values().plot(kind='barh', ax=ax, color='#BA7517')
ax.set_title('Absolut. rozdil dulezitosti: LR vs. Tree modely (normalizovano)', fontsize=12, pad=12)
ax.set_xlabel('|SHAP_tree_avg - SHAP_LR| (normalizovano)')
plt.tight_layout()
plt.show()

Featury s nejvetsi neshodu mezi LR a stromovymi modely:
Semester                  0.285
Gender                    0.250
Scholarship               0.182
Part_Time_Job             0.176
Department_Arts           0.112
Department_Business       0.105
Department_Science        0.101
Department_Engineering    0.091


## ICE heterogenita vs. SHAP důležitost
Zajima nas, zda featury s vysokou SHAP dulezitosti maji take vysokou heterogenitu ICE (tj. rozbihajici se cary).
Pokud ano, jejich efekt je silny A nelinearni/interakcni — dulezite pro interpretaci modelu.
Pokud featura ma vysokou SHAP dulezitost ale nizkou ICE heterogenitu, efekt je homogenni — featura tlaci vsechny studenty stejnym smerem.

In [6]:
from sklearn.inspection import partial_dependence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Prevence FutureWarning: integer sloupce zpusobuji problemy v PDP/ICE
X_test_float = X_test_prep.astype(float)

print("Pocitam ICE heterogenitu (std ICE car)...\n")

# Top 8 featur podle SHAP konsensu — primy vyber z df_shap_norm
top_feats_for_ice = df_shap_norm['SHAP_consensus'].nlargest(8).index.tolist()
print(f"Featury pro ICE: {top_feats_for_ice}")

col_list   = list(X_test_float.columns)
ice_std_rows = []

for name, model in tuned_models.items():
    for feat_name in top_feats_for_ice:
        if feat_name not in col_list:
            print(f"  PRESKAKUJI (neni v datech): {feat_name}")
            continue
        feat_idx = col_list.index(feat_name)
        try:
            pd_result = partial_dependence(
                model, X_test_float,
                features=[feat_idx],
                kind='individual',
            )
            # sklearn vraci 'individual' jako (n_outputs, n_samples, n_grid)
            # pro binarni klasifikaci: n_outputs=1, bereme [0]
            ice_vals = pd_result['individual'][0]
            if ice_vals.ndim == 3:
                ice_vals = ice_vals[0]  # edge case: nektera verze sklearn wrappuje navic
            ice_centered   = ice_vals - ice_vals[:, [0]]
            heterogenita   = float(ice_centered.std(axis=0).mean())
            shap_konsensus = float(df_shap_norm.loc[feat_name, 'SHAP_consensus']) if feat_name in df_shap_norm.index else 0.0
            ice_std_rows.append({
                'Model': name,
                'Feature': feat_name,
                'ICE_heterogenita': heterogenita,
                'SHAP_konsensus': shap_konsensus,
            })
        except Exception as e:
            print(f"  Preskakuji {feat_name} u {name}: {e}")

df_ice_shap = pd.DataFrame(ice_std_rows)

if df_ice_shap.empty:
    print("VAROVANI: df_ice_shap je prazdny — vsechny featury byly preskoceny.")
    print("Zkontroluj vyse vypsane chybove hlasky.")
else:
    print(f"\nVypocitano {len(df_ice_shap)} kombinaci model x featura")

    # Vizualizace
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, name in zip(axes, tuned_models.keys()):
        subset = df_ice_shap[df_ice_shap['Model'] == name]
        if subset.empty:
            ax.set_title(f'{name}\n(zadna data)', fontsize=11)
            continue
        ax.scatter(subset['SHAP_konsensus'], subset['ICE_heterogenita'],
                   s=80, color='#009988', alpha=0.8)
        for _, row in subset.iterrows():
            ax.annotate(row['Feature'], (row['SHAP_konsensus'], row['ICE_heterogenita']),
                        fontsize=8, xytext=(4, 4), textcoords='offset points')
        ax.set_xlabel('SHAP konsensus (dulezitost)')
        ax.set_ylabel('ICE heterogenita (std)')
        ax.set_title(name, fontsize=11)

    fig.suptitle(
    'SHAP důležitost vs. ICE heterogenita efektu\n'
    '(horní pravý kvadrant = vysoká důležitost i nelinearita)',
    fontsize=13, fontweight='bold', y=1.01
)
    plt.tight_layout()
    plt.show()

    threshold_shap = df_ice_shap['SHAP_konsensus'].median()
    threshold_ice  = df_ice_shap['ICE_heterogenita'].median()
    both_high = df_ice_shap[
        (df_ice_shap['SHAP_konsensus'] >= threshold_shap) &
        (df_ice_shap['ICE_heterogenita'] >= threshold_ice)
    ][['Model','Feature','SHAP_konsensus','ICE_heterogenita']].sort_values('SHAP_konsensus', ascending=False)
    print("\nFeatury s vysokou SHAP dulezitosti A vysokou ICE heterogenitou:")
    print(both_high.to_string(index=False))

Pocitam ICE heterogenitu (std ICE car)...

Featury pro ICE: ['CGPA', 'Stress_Index', 'Gender', 'Scholarship', 'Attendance_Rate', 'Semester', 'Part_Time_Job', 'Family_Income']



Vypocitano 32 kombinaci model x featura



Featury s vysokou SHAP dulezitosti A vysokou ICE heterogenitou:
              Model      Feature  SHAP_konsensus  ICE_heterogenita
Logistic Regression         CGPA        1.000000          0.040405
      Random Forest         CGPA        1.000000          0.054904
  Gradient Boosting         CGPA        1.000000          0.067471
      Decision Tree         CGPA        1.000000          0.076548
Logistic Regression Stress_Index        0.250595          0.060763
      Random Forest Stress_Index        0.250595          0.043333
  Gradient Boosting Stress_Index        0.250595          0.069713
      Decision Tree Stress_Index        0.250595          0.054970
      Decision Tree       Gender        0.143039          0.038493
      Decision Tree  Scholarship        0.142832          0.032235


## Shrnutí: konzistentní klíčové featury
Finalni tabulka featur, ktere jsou konzistentne dulezite across SHAP, LIME a Feature Importance.

In [7]:
consensus_top = df_shap_norm['SHAP_consensus'].nlargest(10)
print("=" * 55)
print("  TOP-10 KLICOVYCH FEATUR (SHAP konsensus)")
print("=" * 55)
for i, (feat, score) in enumerate(consensus_top.items(), 1):
    print(f"  {i:2}. {feat:<35} {score:.3f}")
print()
print("Tato zjisteni (SHAP + LIME + ICE) lze pouzit pro:")
print("  - Byznys report (co ovlivnuje dropout?)")
print("  - Pristi iteraci feature engineeringu")
print("  - Uredni zduvodneni rozhodnuti modelu (AI Act)")

  TOP-10 KLICOVYCH FEATUR (SHAP konsensus)
   1. CGPA                                1.000
   2. Stress_Index                        0.251
   3. Gender                              0.143
   4. Scholarship                         0.143
   5. Attendance_Rate                     0.134
   6. Semester                            0.098
   7. Part_Time_Job                       0.066
   8. Family_Income                       0.065
   9. Internet_Access                     0.061
  10. Travel_Time_Minutes                 0.055

Tato zjisteni (SHAP + LIME + ICE) lze pouzit pro:
  - Byznys report (co ovlivnuje dropout?)
  - Pristi iteraci feature engineeringu
  - Uredni zduvodneni rozhodnuti modelu (AI Act)
